<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/13_improve_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, pandas as pd
from tqdm import tqdm
BASE = '/content/drive/MyDrive/longevity_project'
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
compounds = pd.read_csv(f'{BASE}/data/processed/compounds.csv', index_col='name')
sigpca = pd.read_parquet(f'{BASE}/data/processed/signature_pca.parquet')
fp = pd.read_parquet(f'{BASE}/data/processed/ecfp4.parquet')
T = pd.read_parquet(f'{BASE}/data/processed/targets.parquet')
matrix = pd.read_parquet(f'{BASE}/results/hallmark_matrix_baseline.parquet')
holdout = json.load(open(f'{BASE}/data/processed/holdout.json'))
background = json.load(open(f'{BASE}/data/processed/background_negatives.json'))

def similarity(a, b):
    if not (compounds.loc[a, 'has_structure'] and compounds.loc[b, 'has_structure']):
        return 0.0
    x, y = fp.loc[a].values.astype(bool), fp.loc[b].values.astype(bool)
    return (x & y).sum() / (x | y).sum()

def chem_groups(drugs, cutoff=0.6):
    groups = []
    for d in drugs:
        for g in groups:
            if any(similarity(d, m) >= cutoff for m in g):
                g.append(d)
                break
        else:
            groups.append([d])
    return groups

def training_set(hm):
    k = kd['by_hallmark'][hm]
    pos = [d for d in k['helpful'] if d in compounds.index and d not in holdout[hm]]
    neg = [d for d in k['harmful'] if d in compounds.index] + background
    return pos, neg

Mounted at /content/drive


In [2]:
from sklearn.decomposition import TruncatedSVD

has = compounds['has_structure'].values.astype(bool)
fp_small = np.zeros((len(fp), 50), dtype='float32')
fp_small[has] = TruncatedSVD(n_components=50, random_state=0).fit_transform(fp.values[has].astype('float32'))
fpf = pd.DataFrame(fp_small, index=fp.index, columns=[f'fpsvd_{i}' for i in range(50)])
fpf['has_structure'] = has.astype('float32')

FEATURES = {
    'signatures': sigpca,
    'all': pd.concat([sigpca, fpf, T.astype('float32')], axis=1),
}
for name, F in FEATURES.items():
    print(f'{name:11} {F.shape[1]} features')

signatures  100 features
all         419 features


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def make(kind, seed=0):
    if kind == 'logistic':
        return make_pipeline(StandardScaler(),
                             LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000))
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced_subsample',
                                  n_jobs=-1, random_state=seed)

def fit(kind, X, pos, neg, seed=0):
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    return make(kind, seed).fit(X.loc[pos + neg].values, y)

def grouped_cv(kind, feat):
    X, rows = FEATURES[feat], []
    for hm in kd['by_hallmark']:
        pos, neg = training_set(hm)
        groups = chem_groups(pos)
        if len(groups) < 2:
            continue
        for g in groups:
            train_pos = [d for d in pos if d not in g]
            model = fit(kind, X, train_pos, neg)
            rest = X.index.difference(train_pos + neg)
            pct = pd.Series(model.predict_proba(X.loc[rest].values)[:, 1], index=rest).rank(pct=True) * 100
            rows += [{'hallmark': hm, 'drug': d, 'pct': pct[d]} for d in g]
    return pd.DataFrame(rows)

comparison = {}
for kind in ['logistic', 'forest']:
    for feat in ['signatures', 'all']:
        r = grouped_cv(kind, feat)
        comparison[f'{kind}|{feat}'] = r.groupby('hallmark')['pct'].mean()
        print(f'{kind} + {feat} done')

r['baseline'] = [matrix.loc[d, f'{h}_pct'] for h, d in zip(r['hallmark'], r['drug'])]
comparison = pd.DataFrame(comparison)
comparison['baseline'] = r.groupby('hallmark')['baseline'].mean()
comparison.loc['AVERAGE'] = comparison.mean()
print(comparison.round(1))
comparison.round(1).to_csv(f'{BASE}/results/ml_comparison.csv')

logistic + signatures done
logistic + all done
forest + signatures done
forest + all done
                  logistic|signatures  logistic|all  forest|signatures  \
hallmark                                                                 
autophagy                        56.0          27.5               49.1   
inflammation                     47.6          63.4               64.7   
mitochondria                     46.7          67.2               40.7   
nutrient_sensing                 87.9          79.2               72.8   
proteostasis                     75.5          42.6               66.7   
senescence                       40.5          63.2               26.5   
AVERAGE                          59.0          57.2               53.4   

                  forest|all  baseline  
hallmark                                
autophagy               27.5      50.3  
inflammation            68.2      59.1  
mitochondria            37.4      72.3  
nutrient_sensing        56.7      70.8

In [4]:
setups = [c for c in comparison.columns if c != 'baseline']
best = comparison.loc['AVERAGE', setups].idxmax()
kind, feat = best.split('|')
print('Best setup overall:', best, '\n')

decisions = {}
for hm in kd['by_hallmark']:
    ml = comparison[best].get(hm, np.nan)
    base = comparison['baseline'].get(hm, np.nan)
    decisions[hm] = 'ml' if pd.notna(ml) and ml > base + 10 else 'baseline'
    print(f'{hm:17} ML {ml:5.1f} | baseline {base:5.1f} -> use {decisions[hm]}')

json.dump({'setup': best, 'decisions': decisions}, open(f'{BASE}/results/ml_decisions.json', 'w'), indent=2)

Best setup overall: logistic|signatures 

senescence        ML  40.5 | baseline  40.0 -> use baseline
nutrient_sensing  ML  87.9 | baseline  70.8 -> use ml
mitochondria      ML  46.7 | baseline  72.3 -> use baseline
autophagy         ML  56.0 | baseline  50.3 -> use baseline
inflammation      ML  47.6 | baseline  59.1 -> use baseline
proteostasis      ML  75.5 | baseline  65.6 -> use baseline


In [5]:
X = FEATURES[feat]
mean_pct = pd.DataFrame(index=compounds.index)
sd_pct = pd.DataFrame(index=compounds.index)

for hm in tqdm(kd['by_hallmark']):
    pos, neg = training_set(hm)
    rng = np.random.default_rng(1)
    runs = []
    for i in range(25):
        p = list(rng.choice(pos, len(pos), replace=True))
        n = list(rng.choice(neg, len(neg), replace=True))
        model = fit(kind, X, p, n, seed=i)
        runs.append(pd.Series(model.predict_proba(X.values)[:, 1], index=X.index).rank(pct=True) * 100)
    runs = pd.concat(runs, axis=1)
    mean_pct[hm] = runs.mean(axis=1)
    sd_pct[hm] = runs.std(axis=1)

v2 = mean_pct.round(1).add_suffix('_pct').join(sd_pct.round(1).add_suffix('_sd'))
v2.to_parquet(f'{BASE}/results/ml_scores_v2.parquet')

100%|██████████| 6/6 [00:14<00:00,  2.47s/it]


In [6]:
final = pd.DataFrame(index=compounds.index)
for hm, choice in decisions.items():
    final[f'{hm}_pct'] = mean_pct[hm] if choice == 'ml' else matrix[f'{hm}_pct']
    final[f'{hm}_sd'] = sd_pct[hm] if choice == 'ml' else np.nan
    final[f'{hm}_source'] = choice
final['is_named'] = compounds['is_named']
final.round(1).to_parquet(f'{BASE}/results/hallmark_scores_combined.parquet')
print(final.shape)

(15542, 19)


In [7]:
for hm, choice in decisions.items():
    if choice != 'ml':
        continue
    pos, neg = training_set(hm)
    ok = compounds['is_named'] & ~compounds.index.isin(pos + neg)
    top = mean_pct.loc[ok, hm].nlargest(10)
    print(f'{hm:17}', ', '.join(f'{d} ({top[d]:.0f}±{sd_pct.loc[d, hm]:.0f})' for d in top.index))

nutrient_sensing  mitomycin-c (100±0), enzalutamide (100±0), amsacrine (100±0), camptothecin (100±0), dup-697 (100±0), importazole (100±0), aminopurvalanol-a (100±1), ly-294002 (100±1), pi-828 (99±1), sulconazole (99±1)


In [8]:
ml_hms = [h.replace('_', ' ') for h, c in decisions.items() if c == 'ml']
avg = comparison.loc['AVERAGE']
print(f"Compared logistic regression and random forest, with signature-only and signature+fingerprint+target "
      f"features, using grouped cross-validation. Best setup: {best.replace('|', ' with ')} features "
      f"(average left-out percentile {avg[best]:.0f} vs baseline {avg['baseline']:.0f}). "
      f"ML beat the baseline by >10 points for: {', '.join(ml_hms) if ml_hms else 'no hallmarks (baseline kept for all six)'}. "
      f"Final models trained as 25-model ensembles for uncertainty estimates.")

Compared logistic regression and random forest, with signature-only and signature+fingerprint+target features, using grouped cross-validation. Best setup: logistic with signatures features (average left-out percentile 59 vs baseline 60). ML beat the baseline by >10 points for: nutrient sensing. Final models trained as 25-model ensembles for uncertainty estimates.
